# 1. Install MarkLLM

In [1]:
# !pip install -r requirements.txt

download_models = False

model_path = "models/Qwen2.5-Coder-1.5B"

# 2. Watermaring Algorithm Invocation

In [5]:
import torch
import json
from watermark.auto_watermark import AutoWatermark
from utils.transformers_config import TransformersConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load data
with open('dataset/c4/processed_c4.json', 'r') as f:
    lines = f.readlines()
item = json.loads(lines[0])
prompt = item['prompt']
natural_text = item['natural_text']


def test_algorithm(algorithm_name):
    # Check algorithm name
    assert algorithm_name in ['KGW', 'Unigram', 'SWEET', 'EWD', 'SIR', 'XSIR', 'UPV', 'EXP', 'EXPEdit', 'SynthID', 'Adaptive']

    # Device
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Transformers config
    transformers_config = TransformersConfig(model=AutoModelForCausalLM.from_pretrained(model_path).to(device),
                                            tokenizer=AutoTokenizer.from_pretrained(model_path),
                                            vocab_size=50272,
                                            device=device,
                                            max_new_tokens=200,
                                            min_length=230,
                                            do_sample=True,
                                            no_repeat_ngram_size=4)

    # Load watermark algorithm
    myWatermark = AutoWatermark.load(f'{algorithm_name}', algorithm_config=f'config/{algorithm_name}.json', transformers_config=transformers_config)

    # Generate text
    watermarked_text = myWatermark.generate_watermarked_text(prompt)
    unwatermarked_text = myWatermark.generate_unwatermarked_text(prompt)

    # Detect
    detect_result1 = myWatermark.detect_watermark(watermarked_text)
    detect_result2 = myWatermark.detect_watermark(unwatermarked_text)
    detect_result3 = myWatermark.detect_watermark(natural_text)

    print("LLM-generated watermarked text:")
    print(watermarked_text)
    print('\n')
    print(detect_result1)
    print('\n')

    print("LLM-generated unwatermarked text:")
    print(unwatermarked_text)
    print('\n')
    print(detect_result2)
    print('\n')

    print("Natural text:")
    print(natural_text)
    print('\n')
    print(detect_result3)

In [6]:
test_algorithm('KGW')

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
/home/ivan/Projects/MarkLLM/.venv/lib/python3.13/site-packages/transformers/generation/utils.py:1593: UserWarning: Unfeasible length constraints: `min_length` (230) is larger than the maximum possible length (229). Generation will stop at the defined maximum length. You should decrease the minimum length and/or increase the maximum length. Note that `max_length` is set to 229, its default value.
  warnings.warn(
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


LLM-generated watermarked text:
Cluster comprises IBM's Opteron-based eServer 325 server and systems management software and storage devices that can run Linux and Windows operating systems.
The price, not factoring in the software, of the hardware and software equals about US $120,000. The idea is that hardware and OS like it are more important than software, at least for IBM.
The price includes systems software and an array of storage devices such as Fibre Channel, iSCSI and parallel file-system devices. It can serve as a midtier for storage.
You use the midtier software to build out the cluster to higher-peak demands, such as in a Web server area. You can tap the high-tech cluster like a tier-1 SAN. You can do this for various kinds of workloads. So IBM is using Linux to build a product that can serve on top of existing and emerging servers and operating systems.
You could use the IBM cluster or the Apple Xserve in a Web farm. You can use the IBM with Oracle, the open-source and Lin

In [ ]:
test_algorithm('Unigram')

In [ ]:
test_algorithm('SWEET')

In [ ]:
test_algorithm('UPV')

In [ ]:
test_algorithm('EWD')

In [ ]:
test_algorithm('SIR')

In [ ]:
test_algorithm('XSIR')

In [ ]:
test_algorithm('SynthID')

In [ ]:
test_algorithm('EXP')

In [ ]:
test_algorithm('Adaptive')

# 3. Mechanism Visualization

In [ ]:
import torch
import gc
import json
from visualize.font_settings import FontSettings
from watermark.auto_watermark import AutoWatermark
from utils.transformers_config import TransformersConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
from visualize.page_layout_settings import PageLayoutSettings
from visualize.data_for_visualization import DataForVisualization
from visualize.visualizer import DiscreteVisualizer, ContinuousVisualizer
from visualize.legend_settings import DiscreteLegendSettings, ContinuousLegendSettings
from visualize.color_scheme import ColorSchemeForDiscreteVisualization, ColorSchemeForContinuousVisualization
from IPython.display import display
from PIL import Image, ImageOps, ImageFont, ImageDraw

def test_discreet_visualization():
    tokens = ["PREFIX", "PREFIX", "Hello", "world", "this", "is", "a", "test", "Hello", "world", "this", "is", "a", "test", "Hello", "world", "this", "is", "a", "test", "Hello", "world", "this", "is", "a", "test", "Hello", "world", "this", "is", "a", "test"]
    flags = [-1, -1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0]
    weights = [0, 0, 0.1, 0.5, 0.3, 0.8, 0.2, 0.4, 0.1, 0.5, 0.3, 0.8, 0.2, 0.4, 0.1, 0.5, 0.3, 0.8, 0.2, 0.4, 0.1, 0.5, 0.3, 0.8, 0.2, 0.4, 0.1, 0.5, 0.3, 0.8, 0.2, 0.4]

    discreet_visualizer = DiscreteVisualizer(color_scheme=ColorSchemeForDiscreteVisualization(),
                                            font_settings=FontSettings(),
                                            page_layout_settings=PageLayoutSettings(),
                                            legend_settings=DiscreteLegendSettings())
    img = discreet_visualizer.visualize(data=DataForVisualization(tokens, flags, weights),
                                     show_text=True, visualize_weight=True, display_legend=True)

    img.save("test1.png")
    display(img)


def test_continuous_visualization():
    tokens = ["PREFIX", "PREFIX", "Hello", "world", "this", "is", "a", "test", "Hello", "world", "this", "is", "a", "test", "Hello", "world", "this", "is", "a", "test", "Hello", "world", "this", "is", "a", "test", "Hello", "world", "this", "is", "a", "test"]
    values = [None, None, 0.1, 0.5, 0.3, 0.8, 0.2, 0.4, 0.1, 0.5, 0.3, 0.8, 0.2, 0.4, 0.1, 0.5, 0.3, 0.8, 0.2, 0.4, 0.1, 0.5, 0.3, 0.8, 0.2, 0.4, 0.1, 0.5, 0.3, 0.8, 0.2, 0.4]
    weights = [0, 0, 0.1, 0.5, 0.3, 0.8, 0.2, 0.4, 0.1, 0.5, 0.3, 0.8, 0.2, 0.4, 0.1, 0.5, 0.3, 0.8, 0.2, 0.4, 0.1, 0.5, 0.3, 0.8, 0.2, 0.4, 0.1, 0.5, 0.3, 0.8, 0.2, 0.4]

    continuous_visualizer = ContinuousVisualizer(color_scheme=ColorSchemeForContinuousVisualization(),
                                                    font_settings=FontSettings(),
                                                    page_layout_settings=PageLayoutSettings(),
                                                    legend_settings=ContinuousLegendSettings())
    img = continuous_visualizer.visualize(data=DataForVisualization(tokens, values, weights),
                                        show_text=False, visualize_weight=False, display_legend=True)

    img.save("test2.png")
    display(img)


def get_data(algorithm_name):
    # Load data
    with open('dataset/c4/processed_c4.json', 'r') as f:
        lines = f.readlines()
    item = json.loads(lines[0])
    prompt = item['prompt']

    # Device
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Transformers config
    transformers_config = TransformersConfig(model=AutoModelForCausalLM.from_pretrained(model_path).to(device),
                                            tokenizer=AutoTokenizer.from_pretrained(model_path),
                                            vocab_size=50272,
                                            device=device,
                                            max_new_tokens=200,
                                            min_length=230,
                                            no_repeat_ngram_size=4)

    myWatermark = AutoWatermark.load(f'{algorithm_name}',
                                     algorithm_config=f'config/{algorithm_name}.json',
                                     transformers_config=transformers_config)

    watermarked_text = myWatermark.generate_watermarked_text(prompt)
    unwatermarked_text = myWatermark.generate_unwatermarked_text(prompt)
    watermarked_data = myWatermark.get_data_for_visualization(watermarked_text)
    unwatermarked_data = myWatermark.get_data_for_visualization(unwatermarked_text)

    return watermarked_data, unwatermarked_data


def test_visualization_without_weight(algorithm_name, visualize_type='discrete'):
    # Validate input
    assert visualize_type in ['discrete', 'continuous']
    if visualize_type == 'discrete':
        assert algorithm_name in ['KGW', 'Unigram', 'SWEET', 'UPV', 'SIR', 'XSIR', 'EWD']
    else:
        assert algorithm_name in ['EXP', 'EXPEdit']

    # Get data for visualization
    watermarked_data, unwatermarked_data = get_data(algorithm_name)

    # Init visualizer
    if visualize_type == 'discrete':
        visualizer = DiscreteVisualizer(color_scheme=ColorSchemeForDiscreteVisualization(),
                                        font_settings=FontSettings(),
                                        page_layout_settings=PageLayoutSettings(),
                                        legend_settings=DiscreteLegendSettings())
    else:
        visualizer = ContinuousVisualizer(color_scheme=ColorSchemeForContinuousVisualization(),
                                        font_settings=FontSettings(),
                                        page_layout_settings=PageLayoutSettings(),
                                        legend_settings=ContinuousLegendSettings())

    # Visualize
    watermarked_img = visualizer.visualize(data=watermarked_data,
                                           show_text=True,
                                           visualize_weight=True,
                                           display_legend=True)

    unwatermarked_img = visualizer.visualize(data=unwatermarked_data,
                                             show_text=True,
                                             visualize_weight=True,
                                             display_legend=True)

    watermarked_img.save(f"{algorithm_name}_watermarked.png")
    unwatermarked_img.save(f"{algorithm_name}_unwatermarked.png")

    watermarked_width, watermarked_height = watermarked_img.size
    unwatermarked_width, unwatermarked_height = unwatermarked_img.size

    font_size = 22
    font = ImageFont.truetype("./font/times.ttf", font_size)
    title_height = 80

    new_watermarked_img = Image.new('RGB', (watermarked_width, watermarked_height + title_height), (255, 255, 255))
    new_unwatermarked_img = Image.new('RGB', (unwatermarked_width, watermarked_height + title_height), (255, 255, 255))

    draw1 = ImageDraw.Draw(new_watermarked_img)
    text_bbox1 = draw1.textbbox((0, 0), f"{algorithm_name} watermarked", font=font)
    draw1.text((int((watermarked_width - text_bbox1[2] - text_bbox1[0]) * 0.3), int(title_height * 0.35)), f"{algorithm_name} watermarked", fill=(0, 0, 0), font=font)

    draw2 = ImageDraw.Draw(new_unwatermarked_img)
    text_bbox2 = draw2.textbbox((0, 0), f"{algorithm_name} unwatermarked", font=font)
    draw2.text((int((unwatermarked_width - text_bbox2[2] - text_bbox2[0]) * 0.3), int(title_height * 0.35)), f"{algorithm_name} unwatermarked", fill=(0, 0, 0), font=font)

    new_watermarked_img.paste(watermarked_img, (0, title_height))
    new_unwatermarked_img.paste(unwatermarked_img, (0, title_height))

    total_width = watermarked_width + unwatermarked_width
    max_height = watermarked_height + title_height

    new_img = Image.new('RGB', (total_width, max_height))

    new_img.paste(new_watermarked_img, (0, 0))
    new_img.paste(new_unwatermarked_img, (watermarked_width, 0))
    display(new_img)

def test_visualization_with_weight(algorithm_name):
    # Validate input
    assert algorithm_name in ['SWEET', 'EWD']

    # Get data for visualization
    watermarked_data, unwatermarked_data = get_data(algorithm_name)

    # Init visualizer
    visualizer = DiscreteVisualizer(color_scheme=ColorSchemeForDiscreteVisualization(),
                                    font_settings=FontSettings(),
                                    page_layout_settings=PageLayoutSettings(),
                                    legend_settings=DiscreteLegendSettings())

    # Visualize
    watermarked_img = visualizer.visualize(data=watermarked_data,
                                           show_text=True,
                                           visualize_weight=True,
                                           display_legend=True)

    unwatermarked_img = visualizer.visualize(data=unwatermarked_data,
                                             show_text=True,
                                             visualize_weight=True,
                                             display_legend=True)

    watermarked_img.save(f"{algorithm_name}_watermarked.png")
    unwatermarked_img.save(f"{algorithm_name}_unwatermarked.png")

    watermarked_width, watermarked_height = watermarked_img.size
    unwatermarked_width, unwatermarked_height = unwatermarked_img.size

    font_size = 22
    font = ImageFont.truetype("./font/times.ttf", font_size)
    title_height = 80

    new_watermarked_img = Image.new('RGB', (watermarked_width, watermarked_height + title_height), (255, 255, 255))
    new_unwatermarked_img = Image.new('RGB', (unwatermarked_width, watermarked_height + title_height), (255, 255, 255))

    draw1 = ImageDraw.Draw(new_watermarked_img)
    text_bbox1 = draw1.textbbox((0, 0), f"{algorithm_name} watermarked", font=font)
    draw1.text((int((watermarked_width - text_bbox1[2] - text_bbox1[0]) * 0.3), int(title_height * 0.35)), f"{algorithm_name} watermarked", fill=(0, 0, 0), font=font)

    draw2 = ImageDraw.Draw(new_unwatermarked_img)
    text_bbox2 = draw2.textbbox((0, 0), f"{algorithm_name} unwatermarked", font=font)
    draw2.text((int((unwatermarked_width - text_bbox2[2] - text_bbox2[0]) * 0.3), int(title_height * 0.35)), f"{algorithm_name} unwatermarked", fill=(0, 0, 0), font=font)

    new_watermarked_img.paste(watermarked_img, (0, title_height))
    new_unwatermarked_img.paste(unwatermarked_img, (0, title_height))

    total_width = watermarked_width + unwatermarked_width
    max_height = watermarked_height + title_height

    new_img = Image.new('RGB', (total_width, max_height))

    new_img.paste(new_watermarked_img, (0, 0))
    new_img.paste(new_unwatermarked_img, (watermarked_width, 0))
    display(new_img)

## 3.1 Warm Up: Test Visualizer Using Provided Text

In [ ]:
test_discreet_visualization()

In [ ]:
test_continuous_visualization()

## 3.2 Mechansim Visualization of Watermarking Algorithms

### 3.2.1 KGW Family

In [ ]:
test_visualization_without_weight('KGW', 'discrete')

### 3.2.2 Christ Family

In [ ]:
test_visualization_without_weight('EXP', 'continuous')

### 3.2.3 Handling Weighted Token Difference in Visualization

In [ ]:
test_visualization_with_weight('SWEET')

In [ ]:
test_visualization_with_weight('EWD')

# 4. Automated Comprehensive Evaluation

## 4.1 Watermark Detection Pipeline

In [ ]:
import torch
import json
from evaluation.dataset import C4Dataset
from watermark.auto_watermark import AutoWatermark
from utils.transformers_config import TransformersConfig
from evaluation.tools.success_rate_calculator import DynamicThresholdSuccessRateCalculator
from transformers import AutoModelForCausalLM, AutoTokenizer, T5Tokenizer, T5ForConditionalGeneration, BertTokenizer, BertForMaskedLM
from evaluation.pipelines.detection import WatermarkedTextDetectionPipeline, UnWatermarkedTextDetectionPipeline, DetectionPipelineReturnType
from evaluation.tools.text_editor import TruncatePromptTextEditor, TruncateTaskTextEditor, WordDeletion, SynonymSubstitution, ContextAwareSynonymSubstitution, GPTParaphraser, DipperParaphraser

device = 'cuda' if torch.cuda.is_available() else 'cpu'

if torch.cuda.is_available():
  gc.collect()
  torch.cuda.empty_cache()
  with torch.no_grad():
      torch.cuda.empty_cache()

def test_detection_pipeline(algorithm_name, attack_name):
    my_dataset = C4Dataset('dataset/c4/processed_c4.json')
    transformers_config = TransformersConfig(model=AutoModelForCausalLM.from_pretrained(model_path).to(device),
                                             tokenizer=AutoTokenizer.from_pretrained(model_path),
                                             vocab_size=50272,
                                             device=device,
                                             max_new_tokens=200,
                                             min_length=230,
                                             do_sample=True,
                                             no_repeat_ngram_size=4)

    my_watermark = AutoWatermark.load(f'{algorithm_name}', algorithm_config=f'config/{algorithm_name}.json', transformers_config=transformers_config)

    if attack_name == 'Word-D':
      attack = WordDeletion(ratio=0.3)
    elif attack_name == 'Word-S':
      attack = SynonymSubstitution(ratio=0.5)
    elif attack_name == 'Word-S(Context)':
      attack = ContextAwareSynonymSubstitution(ratio=0.5,
                                               tokenizer=BertTokenizer.from_pretrained('/data2/shared_model/bert-large-uncased'),
                                               model=BertForMaskedLM.from_pretrained('/data2/shared_model/bert-large-uncased').to(device))
    elif attack_name == 'Doc-P(GPT-3.5)':
        attack = GPTParaphraser(openai_model='gpt-3.5-turbo',
                                prompt='Please rewrite the following text: ')
    elif attack_name == 'Doc-P(Dipper)':
        attack = DipperParaphraser(tokenizer=T5Tokenizer.from_pretrained('/data2/shared_model/google/t5-v1_1-xxl/'),
                                   model=T5ForConditionalGeneration.from_pretrained('/data2/shared_model/kalpeshk2011/dipper-paraphraser-xxl/', device_map='auto'),
                                   lex_diversity=60, order_diversity=0, sent_interval=1,
                                   max_new_tokens=100, do_sample=True, top_p=0.75, top_k=None)


    pipline1 = WatermarkedTextDetectionPipeline(dataset=my_dataset, text_editor_list=[TruncatePromptTextEditor(), attack],
                                                show_progress=True, return_type=DetectionPipelineReturnType.SCORES)

    pipline2 = UnWatermarkedTextDetectionPipeline(dataset=my_dataset, text_editor_list=[TruncatePromptTextEditor()],
                                            show_progress=True, return_type=DetectionPipelineReturnType.SCORES)

    calculator = DynamicThresholdSuccessRateCalculator(labels=['TPR', 'F1'], rule='best')
    print(calculator.calculate(pipline1.evaluate(my_watermark), pipline2.evaluate(my_watermark)))



In [ ]:
test_detection_pipeline('KGW', 'Word-D')

## 4.2 Text Quality Analysis Pipeline

In [ ]:
import torch
import json
from watermark.auto_watermark import AutoWatermark
from utils.transformers_config import TransformersConfig
from evaluation.dataset import C4Dataset, WMT16DE_ENDataset, HumanEvalDataset
from evaluation.tools.success_rate_calculator import DynamicThresholdSuccessRateCalculator
from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM, AutoTokenizer, LlamaTokenizer
from evaluation.tools.text_editor import TruncatePromptTextEditor, TruncateTaskTextEditor ,CodeGenerationTextEditor
from evaluation.tools.text_quality_analyzer import PPLCalculator, LogDiversityAnalyzer, BLEUCalculator, PassOrNotJudger, GPTTextDiscriminator
from evaluation.pipelines.detection import WatermarkedTextDetectionPipeline, UnWatermarkedTextDetectionPipeline, DetectionPipelineReturnType
from evaluation.pipelines.quality_analysis import (DirectTextQualityAnalysisPipeline, ReferencedTextQualityAnalysisPipeline, ExternalDiscriminatorTextQualityAnalysisPipeline,
                                                   QualityPipelineReturnType)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

if torch.cuda.is_available():
  gc.collect()
  torch.cuda.empty_cache()
  with torch.no_grad():
      torch.cuda.empty_cache()

def test_direct_quality_analysis_pipeline(algorithm_name, quality_analyzer_name):
    my_dataset = C4Dataset('dataset/c4/processed_c4.json')
    transformers_config = TransformersConfig(model=AutoModelForCausalLM.from_pretrained(model_path).to(device),
                          tokenizer=AutoTokenizer.from_pretrained(model_path),
                          vocab_size=50272,
                          device=device,
                          max_new_tokens=200,
                          min_length=230,
                          do_sample=True,
                          no_repeat_ngram_size=4)
    my_watermark = AutoWatermark.load(f'{algorithm_name}', algorithm_config=f'config/{algorithm_name}.json', transformers_config=transformers_config)

    if quality_analyzer_name == 'PPL':
      analyzer = PPLCalculator(model=AutoModelForCausalLM.from_pretrained(llama_path, device_map='auto'),
                                tokenizer=LlamaTokenizer.from_pretrained(llama_path),
                                device=device)
    elif quality_analyzer_name == 'Log Diversity':
      analyzer = LogDiversityAnalyzer()

    analyzers = []
    analyzers.append(analyzer)

    quality_pipeline = DirectTextQualityAnalysisPipeline(dataset=my_dataset,
                                watermarked_text_editor_list=[TruncatePromptTextEditor()],
                                unwatermarked_text_editor_list=[],
                                analyzers=analyzers,
                                unwatermarked_text_source='natural', show_progress=True,
                                return_type=QualityPipelineReturnType.MEAN_SCORES)

    print(f"{quality_analyzer_name}:")
    print(quality_pipeline.evaluate(my_watermark))


def test_referenced_quality_analysis_pipeline_1(algorithm_name):
    """Evaluate the impact on text quality in the machine translation task."""
    my_dataset = WMT16DE_ENDataset('dataset/wmt16_de_en/validation.jsonl')
    tokenizer= AutoTokenizer.from_pretrained(nllb_path, src_lang="deu_Latn")
    transformers_config = TransformersConfig(model=AutoModelForSeq2SeqLM.from_pretrained(nllb_path).to(device),
                                                tokenizer=tokenizer,
                                                device=device,
                                                forced_bos_token_id=tokenizer.lang_code_to_id["eng_Latn"])

    my_watermark = AutoWatermark.load(f'{algorithm_name}', algorithm_config=f'config/{algorithm_name}.json', transformers_config=transformers_config)

    quality_pipeline = ReferencedTextQualityAnalysisPipeline(dataset=my_dataset,
                                                              watermarked_text_editor_list=[],
                                                              unwatermarked_text_editor_list=[],
                                                              analyzer=BLEUCalculator(),
                                                              unwatermarked_text_source='generated', show_progress=True,
                                                              return_type=QualityPipelineReturnType.MEAN_SCORES)


    print("BLEU:")
    print(quality_pipeline.evaluate(my_watermark))


def test_referenced_quality_analysis_pipeline_2(algorithm_name):
    """Evaluate the impact on text quality in the code generation task."""
    my_dataset = HumanEvalDataset('dataset/human_eval/test.jsonl')
    tokenizer= AutoTokenizer.from_pretrained(tiny_starcoder_path)
    transformers_config = TransformersConfig(model=AutoModelForCausalLM.from_pretrained(tiny_starcoder_path, device_map='auto'),
                                             tokenizer=tokenizer,
                                             device=device,
                                             min_length=200,
                                             max_length=400)

    my_watermark = AutoWatermark.load(f'{algorithm_name}', algorithm_config=f'config/{algorithm_name}.json', transformers_config=transformers_config)

    quality_pipeline = ReferencedTextQualityAnalysisPipeline(dataset=my_dataset,
                                  watermarked_text_editor_list=[TruncateTaskTextEditor(),CodeGenerationTextEditor()],
                                  unwatermarked_text_editor_list=[TruncateTaskTextEditor(), CodeGenerationTextEditor()],
                                  analyzer=PassOrNotJudger(),
                                  unwatermarked_text_source='generated', show_progress=True,
                                  return_type=QualityPipelineReturnType.MEAN_SCORES)

    print("pass@1:")
    print(quality_pipeline.evaluate(my_watermark))


def test_discriminator_quality_analysis_pipeline(algorithm_name):
    my_dataset = WMT16DE_ENDataset('dataset/wmt16_de_en/validation.jsonl')
    tokenizer= AutoTokenizer.from_pretrained(nllb_path, src_lang="deu_Latn")
    transformers_config = TransformersConfig(model=AutoModelForSeq2SeqLM.from_pretrained(nllb_path).to(device),
                                                tokenizer=tokenizer,
                                                device=device,
                                                forced_bos_token_id=tokenizer.lang_code_to_id["eng_Latn"])

    my_watermark = AutoWatermark.load(f'{algorithm_name}', algorithm_config=f'config/{algorithm_name}.json', transformers_config=transformers_config)

    quality_pipeline = ExternalDiscriminatorTextQualityAnalysisPipeline(dataset=my_dataset,
                                      watermarked_text_editor_list=[],
                                      unwatermarked_text_editor_list=[],
                                      analyzer=GPTTextDiscriminator(openai_model='gpt-4',
                                      task_description='Translate the following German text to English'),
                                      unwatermarked_text_source='generated', show_progress=True,
                                      return_type=QualityPipelineReturnType.MEAN_SCORES
                                      )

    print("Win Rate:")
    print(quality_pipeline.evaluate(my_watermark))

In [ ]:
test_direct_quality_analysis_pipeline('KGW', 'PPL')

In [ ]:
test_direct_quality_analysis_pipeline('Unigram', 'Log Diversity')

In [ ]:
test_referenced_quality_analysis_pipeline_1('SIR')

In [ ]:
test_referenced_quality_analysis_pipeline_2('SWEET')

In [ ]:
import openai

openai.api_key = ""

test_discriminator_quality_analysis_pipeline('EWD')

# 5. Steal Attack For Watermark

In [ ]:
import torch
import json
from watermark.auto_watermark import AutoWatermark
from utils.transformers_config import TransformersConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load data
with open("dataset/c4/processed_c4.json", "r") as f:
    lines = f.readlines()
item = json.loads(lines[0])
prompt = item["prompt"]
natural_text = item["natural_text"]


def test_steal_algorithm(steal_algorithm_name, watermark_algorithm_name):
    # Check algorithm name
    assert steal_algorithm_name in ["STEAL"]
    assert watermark_algorithm_name in [
        "KGW",
        "Unigram",
        "SWEET",
        "EWD",
        "SIR",
        "XSIR",
        "UPV",
        "EXP",
        "EXPEdit",
        "SynthID",
        "Adaptive",
    ]

    # Device
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Transformers config
    transformers_config = TransformersConfig(
        model=AutoModelForCausalLM.from_pretrained(model_path).to(device),
        tokenizer=AutoTokenizer.from_pretrained(model_path),
        vocab_size=50272,
        device=device,
        max_new_tokens=200,
        min_length=230,
        do_sample=True,
        no_repeat_ngram_size=4,
    )

    # Load steal algorithm and watermark algorithm
    mySteal = AutoWatermark.load(
        f"{steal_algorithm_name}",
        algorithm_config=f"config/{steal_algorithm_name}.json",
        transformers_config=transformers_config,
    )
    myWatermark = AutoWatermark.load(
        f"{watermark_algorithm_name}",
        algorithm_config=f"config/{watermark_algorithm_name}.json",
        transformers_config=transformers_config,
    )

    # Generate text
    steal_text = mySteal.generate_watermarked_text(prompt)
    watermarked_text = myWatermark.generate_watermarked_text(prompt)

    # Detect
    detect_result1 = myWatermark.detect_watermark(steal_text)
    detect_result2 = myWatermark.detect_watermark(watermarked_text)
    detect_result3 = myWatermark.detect_watermark(natural_text)

    print("Watermarked text after Stolen and Scrubbing:")
    print(steal_text)
    print("\n")
    print(detect_result1)
    print("\n")

    print("LLM-generated watermarked text:")
    print(watermarked_text)
    print("\n")
    print(detect_result2)
    print("\n")

    print("Natural text:")
    print(natural_text)
    print("\n")
    print(detect_result3)

In [ ]:
test_steal_algorithm("STEAL", "KGW")

# 6. ETH Publicly Detectable Watermark

Embeds an Ethereum ECDSA signature of the generated prefix into subsequent text using a chain of SHA-256 hashes over fixed-length character segments (adapted from [Fairoze et al.](publicly-detectable-watermark/)). Detection recovers the signer's ETH address from the text without any secret key.

In [2]:
import torch
import json
from watermark.auto_watermark import AutoWatermark
from utils.transformers_config import TransformersConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"

# Same C4 prompt as section 2 (other watermark demos)
with open('dataset/c4/processed_c4.json', 'r') as f:
    lines = f.readlines()
item = json.loads(lines[0])
prompt = item['prompt']

transformers_config = TransformersConfig(
    model=AutoModelForCausalLM.from_pretrained(
        model_path, trust_remote_code=True
    ).to(device),
    tokenizer=AutoTokenizer.from_pretrained(model_path, trust_remote_code=True),
    vocab_size=151936,
    device=device,
    max_new_tokens=200,
    do_sample=True,
    no_repeat_ngram_size=4,
)

eth_watermark = AutoWatermark.load(
    'ETHWatermark',
    algorithm_config='config/ETHWatermark.json',
    transformers_config=transformers_config,
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [3]:
from eth_account import Account

# Create a fresh ETH account for signing
eth_account = Account.create()
print(f"Signer ETH address: {eth_account.address}")

# Generate watermarked text (the signature is embedded in the output)
watermarked_text = eth_watermark.generate_watermarked_text(
    prompt, private_key=eth_account.key,
)

print(f"\nWatermarked text:\n{watermarked_text}")

Signer ETH address: 0x5f028A805F8731b1960A63d6F8bd0eF91D048B08


Embedding signature: 100%|████████████████████| 260/260 [05:22<00:00,  1.24s/it]



Watermarked text:
. The system includes a two-processor, four-core Intel Xeon processor card I has an4 second systemandandaax onan , windowson from comprisesystemcom comes contains several come are company the, windows core Software more Win in window other aSystem form company Form O Cluster is the buffer support computer comprises Com re's = m card operate - os he。

 my process view manager about processor bank set use i man core ip me linux mod opt storage license fan end machine than o set w ont how IBM dos sum let & j assembly manage M is- am s y management force own function an in common have manages monitor wasd min consider she. sam sys many cIBM as come format off from what her be servers device sein daemon d ash som admin span  weserver som systems com then some do server (s mon om bor am ran es canon has will stem store japan not screen stand organization manager disk coma ein per extend for standard; span ag an button system list can ones ind


 one libs equipment into men

In [4]:
# Detect watermark: recover the signer address from the text
detect_result = eth_watermark.detect_watermark(watermarked_text)
print("Detection on watermarked text:")
print(detect_result)

# Verify the recovered address matches the signer
if detect_result["recovered_address"]:
    match = detect_result["recovered_address"].lower() == eth_account.address.lower()
    print(f"Address match: {match}")

# Detection on unwatermarked text (should find no watermark)
unwatermarked_text = eth_watermark.generate_unwatermarked_text(prompt)
print(f"\nUnwatermarked text:\n{unwatermarked_text}")
detect_result_unwatermarked = eth_watermark.detect_watermark(unwatermarked_text)
print(f"\nDetection on unwatermarked text:\n{detect_result_unwatermarked}")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Detection on watermarked text:
{'is_watermarked': True, 'recovered_address': '0x482f863580919085BF6ea519C82d8E78A087d2A8'}
Address match: True

Unwatermarked text:
Cluster comprises IBM's Opteron-based eServer 325 server and systems management software and storage devices that can run Linux and Windows operating systems, and IBM will distribute its Linux software to the group, said a spokesperson.
At the 25th International Linux Symposium, a U.S. distributor called Red Hat, Inc. of New York, presented a system called Tru64 Linux as one of four finalists for a top 100 award at the Symposium.
The company now offers the $995 Windows-based OS on the IBM Tru6x platform, which is based on the Suse 7.1 Linux package and IBM 32/64-bit architecture running on the new IBM POWER5-based POWER4 servers. The company is also supporting existing Suse 6.2 and 7.3 versions of the Unix-based Linux software and the SUSE Linux Enterprise Server (LESE) 9.1 operating system, it said.

Detection on unwatermar